# Fabrica de GIFs — MapBiomas

Notebook interativo para gerar GIFs animados de dados do MapBiomas via **Google Earth Engine**.

Funciona em:
- **Google Colab** (Pro: 8 workers, RAM ampliada)
- **VS Code** (Jupyter extension, usa venv local)

---

## Como usar

1. Execute a Celula 1 (Setup)
2. Na Celula 2, selecione dataset, produtos e territorios nas abas
3. Execute a Celula 3 para disparar o batch
4. Veja os resultados na Celula 4

Dica: pode re-executar a Celula 3 quantas vezes quiser sem refazer a interface.

In [ ]:
# @title Celula 1: Setup & Ambiente

import sys, os, multiprocessing, glob, re, shutil, threading
from datetime import datetime
from collections import defaultdict

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    !pip install -q earthengine-api pillow pyyaml ipywidgets 2>/dev/null
    !git clone -q https://github.com/wallyboy22/gif_factory.git 2>/dev/null || true
    %cd gif_factory
    import ee
    ee.Authenticate()
    ee.Initialize(project='ee-ipam')
else:
    cwd = os.getcwd()
    sys.path.insert(0, cwd)
    if not os.path.exists(os.path.join(cwd, 'src')):
        parent = os.path.dirname(cwd)
        if os.path.exists(os.path.join(parent, 'src')):
            sys.path.insert(0, parent)
            os.chdir(parent)
    import ee
    try:
        ee.Initialize(project='ee-ipam')
    except Exception:
        pass

from src.ipam_gif_factory.config import ConfigLoader
from src.ipam_gif_factory.core.pipeline import Pipeline
from src.ipam_gif_factory.interfaces.ui_components import (
    PipelineStepUI, make_spinner, make_select_all_none,
    make_sync_button, make_empty_state, inline_confirm
)

import ipywidgets as widgets
from IPython.display import display, clear_output

config = ConfigLoader()
WORKERS = max(1, min((os.cpu_count() or 4) - 1, 12))

# --- Cache: escaneia outputs/ por GIFs ja gerados ---

gif_cache = defaultdict(set)

def build_gif_cache():
    """Escaneia outputs/v001/ por .gif e .state_gif para saber o que ja existe."""
    global gif_cache
    gif_cache.clear()
    output_base = config.get_output_dir()
    for gif_path in glob.glob(os.path.join(output_base, '**', '*.gif'), recursive=True):
        rel = os.path.relpath(gif_path, output_base).replace('\\', '/')
        parts = rel.split('/')
        if len(parts) >= 3:
            ds, prod, terr = parts[0], parts[1], parts[2]
            gif_cache[(ds, prod)].add(terr)
    # Tambem checa .state_gif (checkpoint de GIF concluido)
    for state_path in glob.glob(os.path.join(output_base, '**', '.state_gif'), recursive=True):
        d = os.path.dirname(state_path)
        rel = os.path.relpath(d, output_base).replace('\\', '/')
        parts = rel.split('/')
        if len(parts) >= 3:
            ds, prod, terr = parts[0], parts[1], parts[2]
            gif_cache[(ds, prod)].add(terr)

build_gif_cache()

def flatten_territories(territories_dict):
    """Achata territories.yaml (nested por grupo) em lista plana de IDs."""
    result = []
    for group_name, group in territories_dict.items():
        if isinstance(group, dict):
            for tid in group.keys():
                result.append(tid)
    return sorted(result)

ALL_TERRITORIES = flatten_territories(config.territories)

TERRITORY_GROUPS = {
    'countries': [], 'biomes': [], 'states': [], 'custom_regions': []
}
for gname, group in config.territories.items():
    if isinstance(group, dict):
        if gname == 'countries':
            TERRITORY_GROUPS['countries'] = sorted(group.keys())
        elif gname == 'biomes':
            TERRITORY_GROUPS['biomes'] = sorted(group.keys())
        elif gname == 'states':
            TERRITORY_GROUPS['states'] = sorted(group.keys())
        elif gname == 'custom_regions':
            TERRITORY_GROUPS['custom_regions'] = sorted(group.keys())

DATASET_CATEGORIES = {}
for ds_id, ds_data in config.datasets.items():
    cat = ds_data.get('category', 'other')
    if cat not in DATASET_CATEGORIES:
        DATASET_CATEGORIES[cat] = []
    DATASET_CATEGORIES[cat].append(ds_id)

print(f"Ambiente: {'Google Colab' if IN_COLAB else 'VS Code'}")
print(f"Workers auto-detectados: {WORKERS}")
print(f"Datasets: {len(list(config.datasets.keys()))} | Territorios: {len(ALL_TERRITORIES)}")
print(f"Cache: {sum(len(v) for v in gif_cache.values())} GIFs encontrados")
print("Pronto!")

In [ ]:
# @title Celula 2: Interface de Selecao

# === Estado global da UI ===
ui_products = {}       # {(dataset_id, product_id): checkbox}
ui_territories = {}    # {territory_id: checkbox}

# === UI Principal ===
ui = PipelineStepUI(title="Fabrica de GIFs", description="Selecione dataset, produtos e territorios.")

# --- Abas de Dataset ---
category_tabs = widgets.Tab()
category_names = sorted(DATASET_CATEGORIES.keys())
category_children = []

def make_dataset_grid(category):
    """Cria grid de checkboxes para os datasets de uma categoria."""
    ds_ids = sorted(DATASET_CATEGORIES.get(category, []))
    if not ds_ids:
        return make_empty_state("Nenhum dataset nesta categoria.")

    prod_by_ds = {}
    for ds_id in ds_ids:
        ds_data = config.datasets.get(ds_id, {})
        prod_by_ds[ds_id] = sorted(ds_data.get('products', {}).keys())

    all_chks = []
    rows = []
    for ds_id in ds_ids:
        prods = prod_by_ds[ds_id]
        ds_label = widgets.HTML(
            f'<div style="font-weight:bold; margin:8px 0 4px 0; color:#333;">{ds_id}</div>')
        rows.append(ds_label)

        chk_row = []
        for prod_id in prods:
            key = (ds_id, prod_id)
            exists = prod_id in gif_cache.get((ds_id, prod_id), set()) or len(
                gif_cache.get((ds_id, prod_id), set())) > 0
            chk = widgets.Checkbox(
                value=False, indent=False,
                description=prod_id,
                layout=widgets.Layout(width='320px'),
                disabled=False,
                style={'description_width': 'initial'}
            )
            chk._meta = {'dataset': ds_id, 'product': prod_id, 'exists': False}
            ui_products[key] = chk
            all_chks.append(chk)
            chk_row.append(chk)

        # 2 colunas de checkboxes
        col_size = (len(chk_row) + 1) // 2
        for i in range(0, len(chk_row), col_size):
            batch = chk_row[i:i + col_size]
            while len(batch) < col_size:
                batch.append(widgets.HTML(''))
            rows.append(widgets.HBox(batch, layout=widgets.Layout(margin='2px 10px')))

    def select_all(_):
        for c in all_chks:
            if not c.disabled:
                c.value = True

    def select_none(_):
        for c in all_chks:
            c.value = False

    btns = make_select_all_none(select_all, select_none)
    _, _, btns_row = btns if isinstance(btns, tuple) else (None, None, btns)
    return widgets.VBox([btns_row] + rows,
        layout=widgets.Layout(max_height='450px', overflow_y='auto', padding='5px'))

for i, cat in enumerate(category_names):
    grid = make_dataset_grid(cat)
    category_children.append(grid)
    category_tabs.set_title(i, cat)

category_tabs.children = category_children

# --- Abas de Territorios ---
territory_tabs = widgets.Tab()
territory_group_names = ['countries', 'biomes', 'states', 'custom_regions']
territory_labels = ['Paises', 'Biomas', 'Estados', 'Regioes']
territory_children = []

def make_territory_grid(group_key):
    """Cria grid de checkboxes para um grupo de territorios."""
    tids = TERRITORY_GROUPS.get(group_key, [])
    if not tids:
        return make_empty_state("Nenhum territorio neste grupo.")

    all_chks = []
    chk_list = []
    for tid in tids:
        chk = widgets.Checkbox(
            value=False, indent=False,
            description=tid,
            layout=widgets.Layout(width='180px'),
            style={'description_width': 'initial'}
        )
        ui_territories[tid] = chk
        all_chks.append(chk)
        chk_list.append(chk)

    cols = 4
    rows_widgets = []
    for i in range(0, len(chk_list), cols):
        batch = chk_list[i:i + cols]
        while len(batch) < cols:
            batch.append(widgets.HTML(''))
        rows_widgets.append(widgets.HBox(batch, layout=widgets.Layout(margin='2px 5px')))

    def select_all(_):
        for c in all_chks:
            c.value = True

    def select_none(_):
        for c in all_chks:
            c.value = False

    btns = make_select_all_none(select_all, select_none)
    _, _, btns_row = btns if isinstance(btns, tuple) else (None, None, btns)
    return widgets.VBox([btns_row] + rows_widgets,
        layout=widgets.Layout(max_height='400px', overflow_y='auto', padding='5px'))

for i, gkey in enumerate(territory_group_names):
    grid = make_territory_grid(gkey)
    territory_children.append(grid)
    territory_tabs.set_title(i, territory_labels[i])

territory_tabs.children = territory_children

# --- Configuracao ---
workers_tx = widgets.IntText(value=WORKERS, description='Workers:', layout=widgets.Layout(width='150px'))
resume_cb = widgets.Checkbox(value=True, description='Resume')
collage_cb = widgets.Checkbox(value=True, description='Collage')
dimension_tx = widgets.IntText(value=1560, description='Altura px:', layout=widgets.Layout(width='150px'))
edit_cb = widgets.Checkbox(value=False, description='Modo Edicao (desbloqueia checkboxes)')

config_row = widgets.HBox(
    [workers_tx, resume_cb, collage_cb, dimension_tx, edit_cb],
    layout=widgets.Layout(gap='15px', margin='10px 0', align_items='center'))

# --- Callbacks ---
def on_edit_change(change):
    """Modo edicao: desbloqueia checkboxes para permitir selecao de itens ja gerados."""
    editable = change['new']
    for chk in ui_products.values():
        if chk._meta.get('exists') and not editable:
            chk.disabled = True
            chk.value = False
        else:
            chk.disabled = False
    # Mostra/oculta botoes de delete
    delete_box.layout.display = 'block' if editable else 'none'

edit_cb.observe(on_edit_change, names='value')

def refresh_cache(_):
    build_gif_cache()
    # Reconstroi status nos checkboxes
    for (ds_id, prod_id), chk in ui_products.items():
        chk._meta['exists'] = False
        chk.disabled = False
        chk.value = False
    ui.log(f"Cache atualizado: {sum(len(v) for v in gif_cache.values())} GIFs", "success")

refresh_btn = make_sync_button("Atualizar Cache", refresh_cache, ui=ui, width='180px')

# --- Delete ---
delete_box = widgets.VBox(layout=widgets.Layout(display='none'))
delete_btn = widgets.Button(
    description="Excluir Selecionados", button_style='danger',
    layout=widgets.Layout(width='200px'))

def on_delete_click(b):
    def do_delete():
        output_base = config.get_output_dir()
        deleted = 0
        for (ds_id, prod_id), chk in ui_products.items():
            if chk.value:
                for tid, tchk in ui_territories.items():
                    if tchk.value:
                        path = os.path.join(output_base, ds_id, prod_id, tid)
                        if os.path.exists(path):
                            shutil.rmtree(path)
                            deleted += 1
                            ui.log(f"Excluido: {ds_id}/{prod_id}/{tid}", "warning")
        build_gif_cache()
        ui.log(f"{deleted} diretorios excluidos. Cache atualizado.", "success")
        delete_box.layout.display = 'none'
    inline_confirm(delete_btn, do_delete)

delete_btn.on_click(on_delete_click)
delete_box.children = [delete_btn]

# --- Monta layout ---
ui.main_area.children = [
    PipelineStepUI.get_status_css(),
    widgets.HTML('<h4 style="margin:10px 0 5px 0;">Datasets & Produtos</h4>'),
    category_tabs,
    widgets.HTML('<h4 style="margin:15px 0 5px 0;">Territorios</h4>'),
    territory_tabs,
    config_row,
    widgets.HBox([refresh_btn, delete_box],
                 layout=widgets.Layout(gap='10px', margin='5px 0')),
]
ui.display()
print("Pronto. Selecione os itens e va para a Celula 3.")

In [ ]:
# @title Celula 3: Disparar Batch

from concurrent.futures import ThreadPoolExecutor, as_completed

print_lock = threading.Lock()
results_list = []

def process_one(dataset_id, prod_id, territory_id, resume):
    pipeline = Pipeline(config)
    result = pipeline.run(
        dataset_id=dataset_id,
        product_id=prod_id,
        territory_id=territory_id,
        create_collage=collage_cb.value,
        add_labels=True,
        vertical_dimension=dimension_tx.value,
        cell_height=300,
        resume=resume,
    )
    return result

def start_batch():
    """Le os checkboxes da UI e dispara o batch."""
    selected_products = [
        (ds_id, prod_id)
        for (ds_id, prod_id), chk in ui_products.items()
        if chk.value
    ]
    selected_territories = [
        tid for tid, chk in ui_territories.items() if chk.value
    ]

    if not selected_products:
        ui.log("Selecione pelo menos 1 produto.", "error")
        return
    if not selected_territories:
        ui.log("Selecione pelo menos 1 territorio.", "error")
        return

    combos = [
        (ds_id, prod_id, tid)
        for ds_id, prod_id in selected_products
        for tid in selected_territories
    ]
    total = len(combos)
    workers = workers_tx.value
    resume = resume_cb.value

    ui.clear_logs()
    ui.show_loader(f"Iniciando {total} combos...")
    ui.log(f"Produtos: {len(selected_products)} | Territorios: {len(selected_territories)}", "info")
    ui.log(f"Total: {total} combinacoes | Workers: {workers} | Resume: {resume}", "info")

    global results_list
    results_list.clear()
    ok = 0
    fail = 0

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {
            executor.submit(process_one, ds_id, prod_id, tid, resume): (ds_id, prod_id, tid)
            for ds_id, prod_id, tid in combos
        }
        for i, f in enumerate(as_completed(futures), 1):
            ds_id, prod_id, tid = futures[f]
            result = f.result()
            status = result.get('status', '?')
            results_list.append(result)
            if status == 'success':
                ok += 1
                ui.log(f"[{i}/{total}] {prod_id} / {tid}  OK", "success")
            else:
                fail += 1
                err = result.get('error', '')
                ui.log(f"[{i}/{total}] {prod_id} / {tid}  FALHA: {err}", "error")

    ui.hide_loader()
    ui.log(f"RESUMO: {ok} OK / {fail} Falha / {total} Total", "success" if fail == 0 else "warning")
    build_gif_cache()
    ui.log(f"Cache atualizado: {sum(len(v) for v in gif_cache.values())} GIFs", "info")

# Executa (pode re-executar esta celula quantas vezes quiser)
start_batch()

In [ ]:
# @title Celula 4: Resultados & Preview

from IPython.display import Image as IPImage, display as ipydisplay

output_base = config.get_output_dir()

if results_list:
    print("## Ultimos GIFs gerados:\n")
    for i, r in enumerate(results_list):
        if r['status'] == 'success':
            gif = r.get('gif_path', '')
            if gif and os.path.exists(gif):
                size_mb = os.path.getsize(gif) / (1024 * 1024)
                print(f"{i+1}. {r['product']} / {r['territory']} ({size_mb:.1f} MB)")

    print(f"\n## Preview dos 3 primeiros:\n")
    count = 0
    for r in results_list:
        if count >= 3:
            break
        if r['status'] == 'success' and r.get('gif_path') and os.path.exists(r['gif_path']):
            print(f"### {r['product']} — {r['territory']}")
            ipydisplay(IPImage(filename=r['gif_path']))
            count += 1
else:
    print("Nenhum resultado ainda. Rode a Celula 3 primeiro.")
    print(f"\nProcurando GIFs existentes em {output_base}...")
    gifs = sorted(glob.glob(os.path.join(output_base, '**', '*.gif'), recursive=True))
    print(f"Encontrados: {len(gifs)} GIFs")
    for g in gifs[:5]:
        print(f"  {os.path.relpath(g, output_base)}")